某一間商店有 $N$ 件商品，其售價及成本分別為 $p_n$ 及 $c_n$， $n = 1, \cdots, N$。\
這些商品分別存放於對應的倉庫，倉庫的容量記為 $V_n$， $n = 1, \cdots, N$。\
每天商品的需求量服從 Poisson 分布，其期望值 $\lambda$ 僅依據當天是否為假日 (記為 $\lambda_{n,0}$ :平日；$\lambda_{n,1}$ :假日)。\
$\mathrm{Reward}(x) = -(x - 3)^2$，也就是與目標庫存量的方差，這推測會導致學習的成果與 linear regression 類似。

In [1]:
import numpy as np

In [2]:
# Parameter
N = 3
prices = [50 * n for n in range(1, N + 1)]
costs = [25 * n for n in range(1, N + 1)]
capacity = [5 + n for n in range(1, N + 1)]
lambda1 = [2 * n for n in range(1, N + 1)]
lambda0 = [n for n in range(1, N + 1)]
gamma = 1
isHoliday = {n: False for n in range(7)}
isHoliday[0] = True
isHoliday[6] = True
T = 100 # simulation time

In [3]:
expectRemainder = 3
def rewardValue(x):
    return -(x - expectRemainder) ** 2

In [4]:
states = []
state = [0 for _ in range(N + 1)]
def generateStates(states, state, index):
    if index == N:
        for day in range(7):
            state[index] = day
            states.append(tuple(state.copy()))
        return
    for i in range(capacity[index] + 1):
        state[index] = i
        generateStates(states, state, index + 1)
generateStates(states, state, 0)
statesLen = len(states)

In [5]:
stateActionPair = {state : [] for state in states}
action = [0 for _ in range(N)]
def generateActions(stateActionPair, action, index, state):
    if index == N:
        stateActionPair[state].append(tuple(action.copy()))
        return
    for i in range(capacity[index] - state[index] + 1):
        action[index] = i
        generateActions(stateActionPair, action, index + 1, state)
for state in states:
    generateActions(stateActionPair, action, 0, state)

In [6]:
dim = 0
for state in states:
    dim += len(stateActionPair[state])
print(dim)

317520


In [ ]:
# Initialize
policy = {state : tuple([0 for n in range(N)]) for state in states}
avg = {state : {action : 0 for action in stateActionPair[state]} for state in states}
num = {state : {action : 0 for action in stateActionPair[state]} for state in states}
for t in range(T):
    print("{:3}".format(t+1), end = " ")
    if (t + 1) % 10 == 0:
        print()
    for state in states:
        for action in stateActionPair[state]:
            rewards = []
            visit = []
            currState = state
            currAction = action
            while True:
                # simulation
                visit.insert(0, (currState, currAction))
                nextState = list(currState)
                reward = 0
                for n in range(N):
                    if isHoliday[currState[-1]]:
                        # holiday
                        request = np.random.poisson(lambda1[n])
                    else:
                        # weekday
                        request = np.random.poisson(lambda0[n])
                    remainder = currState[n] + currAction[n] - request
                    reward += rewardValue(remainder)
                    # reward += prices[n] * min(request, state[n] + action[n])
                    # reward -= costs[n] * action[n]
                    nextState[n] = max(remainder, 0)
                rewards.insert(0, reward)
                nextState[-1] += 1
                currState = tuple(nextState)
                if currState[-1] >= 6:
                    break
                currAction = policy[currState]
            G = 0
            for i in range(len(rewards)):
                s, a = visit[i]
                r = rewards[i]
                G = gamma * G + r
                num[s][a] += 1
                avg[s][a] = avg[s][a] + (r - avg[s][a]) / num[s][a]
                if avg[s][a] > avg[s][policy[s]]:
                    policy[s] = a

  1   2   3   4   5   6   7   8   9  10 
 11  12  13  14  15  16  17  18  19  20 
 21  22  23  24  25  26  27  28  29  30 
 31  32  33  34  35  36  37  38  39  40 
 41  42  43  44  45  46  47  48  49  50 
 51  52  53  54  55  56  57  58  59  60 
 61  62  63  64  65  66  67  68  69  70 
 71  72  73  74  75  76  77  78  79  80 
 81  82  83  84  85  86  87  88  89  90 
 91  92  93  94  95  96  97  98  99 100 


In [8]:
sample = np.random.randint(statesLen)
state = states[sample]
print(f"Sample state is {state}, policy is {policy[state]}.")
if isHoliday[state[-1]]:
    expectRequest = lambda1
else:
    expectRequest = lambda0
print(f"Expected request is {tuple(expectRequest)}.")
opitmal = tuple([expectRemainder + exp for exp in expectRequest])
curr = tuple([state[n] + policy[state][n] for n in range(N)])
print(f"Optimal stock number is {opitmal}; current stock number is {curr}.")

Sample state is (4, 2, 0, 5), policy is (0, 3, 6).
Expected request is (1, 2, 3).
Optimal stock number is (4, 5, 6); current stock number is (4, 5, 6).
